<a href="https://colab.research.google.com/github/prithwis/PrashnaSathi/blob/main/PrashnaSathi_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![alt text](https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/images/CentaurBanner2.png)<br>


<hr>

[Prithwis Mukerjee](http://www.linkedin.com/in/prithwis)<br>

#Install PreRequisites and Utilities

Load API key<br>
OPENAI_API_KEY needs to defined as a Colab "secret" for the Google ID used to run this notebook

In [1]:
!pip install --quiet openai

!wget -q -O PrashnaSathi.py https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/utilities/Centaur_v2.py
import PrashnaSathi as ps

API key loaded ✔
Logged in as Calcutta prithwis@yantrajaal.com


Select Model

In [8]:
# ---------------------------------------------------------------------------------------------------------
#| Model            | Best For                  | Notes                                       |
#| ---------------- | ------------------------- | ------------------------------------------- |
#| **GPT-4.1**      | Highest-quality reasoning | Ideal judge for complex scenario evaluation |
#| **GPT-4.1-mini** | Balanced reasoning & cost | Strong choice for adjudication logic        |
#| **GPT-4.1-nano** | High volume, low cost     | Good for simple reasoning tasks             |
#| **gpt-4o-mini**  | Prototyping & cheap       | Great starter, but upgrade recommended      |
# ---------------------------------------------------------------------------------------------------------
cModel = "gpt-4o-mini"
#cModel = "gpt-4.1-mini"


#Define PrashnaSathi : CareerRole


In [2]:
%%writefile CareerRole.txt

You are PrashnaSathi, an AI assistant whose purpose is to help a user formulate
a high-quality prompt for career guidance from another LLM.

Your role is NOT to provide career advice or recommend a career.

Your role is to understand the user's situation by asking intelligent,
relevant and adaptive questions.

The user is either:
- in the final year of an undergraduate degree, or
- has recently completed an undergraduate degree,

and wants to determine what they should do next.

You should progressively understand factors such as:
- educational background
- demonstrated abilities and skills
- interests and dislikes
- work or project experience
- economic circumstances
- family expectations and constraints
- geographical constraints
- willingness and ability to pursue further education
- career aspirations and priorities

Do NOT mechanically work through this list as a questionnaire.

Use information already provided by the user to decide what information is
still missing. Ask questions whose answers would materially improve the
eventual career-guidance prompt.

Ask only a small number of questions at each turn.

Do not provide career recommendations yourself.

When you believe you have sufficient information, say so and construct a
comprehensive prompt that the user can submit to an LLM of their choice for
career guidance.

Writing CareerRole.txt


In [10]:
with open("/content/CareerRole.txt", "r", encoding="utf-8") as f:
    PS_Role = f.read()

# Optional sanity check
print(Career_Role[:200])


You are PrashnaSathi, an AI assistant whose purpose is to help a user formulate
a high-quality prompt for career guidance from another LLM.

Your role is NOT to provide career advice or recommend a c


In [11]:
def nextQuestion(story):

    context = f"""
CURRENT STORY:

{story}

Based on the ROLE, ask the single most useful question that you should ask next.

Return ONLY the question.
"""

    result = ps.OpenAI_llm_call(
        PS_Role,
        context,
        model=cModel
    )

    return result["content"]

In [12]:
story = ""

while True:

    question = nextQuestion(story)

    print("\nPrashnaSathi:", question)

    answer = input("\nYour answer [STOP to finish]: ")

    if answer.strip().upper() == "STOP":
        break

    story += f"""
    Question: {question}
    Answer: {answer.strip()}
    """

print("\n---  STORY ---\n")
print(story)


PrashnaSathi: What is your major or field of study, and what specific subjects or topics within it do you enjoy the most?

Your answer [STOP to finish]: BA English Honours

PrashnaSathi: What skills or strengths do you feel you have developed during your studies in English that you would like to utilize in your future career?

Your answer [STOP to finish]: Write well

PrashnaSathi: What types of work or project experiences have you had during your studies, such as internships, part-time jobs, or extracurricular activities?

Your answer [STOP to finish]: one part time job

PrashnaSathi: What are your career aspirations or goals after completing your degree, and are there specific industries or roles you are particularly interested in?

Your answer [STOP to finish]: STOP

---  STORY ---


    Question: What is your major or field of study, and what specific subjects or topics within it do you enjoy the most?
    Answer: BA English Honours
    
    Question: What skills or strengths do y

In [13]:
def buildStory():

    story = ""

    while True:

        question = nextQuestion(story)

        print("\nPrashnaSathi:", question)

        answer = input("\nYour answer [STOP to finish]: ")

        if answer.strip().upper() == "STOP":
            break

        story += f"""
Question: {question}
Answer: {answer.strip()}
"""

    return story

In [14]:
story = buildStory()
print("\n--- STORY ---\n")
print(story)


PrashnaSathi: What is your major or field of study, and what subjects or topics within it have you enjoyed the most?

Your answer [STOP to finish]: line 1

PrashnaSathi: What specific skills or abilities have you developed during your studies that you feel confident about?

Your answer [STOP to finish]: line 2

PrashnaSathi: What work or project experience do you have related to your field of study, and what did you learn from those experiences?

Your answer [STOP to finish]: quit

PrashnaSathi: What are your interests and dislikes outside of your academic field?

Your answer [STOP to finish]: stop

--- STORY ---


Question: What is your major or field of study, and what subjects or topics within it have you enjoyed the most?
Answer: line 1

Question: What specific skills or abilities have you developed during your studies that you feel confident about?
Answer: line 2

Question: What work or project experience do you have related to your field of study, and what did you learn from tho

In [ ]:
def ZeitWorld(
    world_input: str,
    executed_action: str = None
):
    """
    ZeitWorld — World Synthesizer

    Inputs:
    - world_input:
        Either:
        (a) Raw unstructured narrative describing the world, OR
        (b) A semi-structured faceted world state from a previous turn

    - executed_action:
        A description of an action that has already been executed.
        May be None during world initialization.

    Output:
    - A semi-structured faceted world narrative suitable for subsequent steps.
    """

    ROLE = ZW_Role                  # As read from text file

    context = f"""
    WORLD INPUT:
    {world_input}

    EXECUTED ACTION:
    {executed_action if executed_action else "None (world initialization phase)"}

    Task:
    1. If this is initialization:
    - Extract salient facets from the world input.
    - Name each facet succinctly.
    - Stabilize an initial interpretive world state.

    2. If this is an update:
    - Reflect only the direct and second-order consequences
        of the executed action.
    - Update relevant facets conservatively.
    - Leave unrelated facets unchanged.

    Return the updated world as a faceted narrative state.
    """

    llm_response = cn.OpenAI_llm_call(
        ROLE,
        context,
        model=cModel
    )

    llm_response["header"] = "ZeitWorld | " + llm_response["header"]

    return llm_response


#Initialise World

In [ ]:
#!cat World_0.txt
#!wget -q -O World_0.txt https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/worlds/REEWorld_Simple.txt
!wget -q -O World_0.txt https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/worlds/REEWorld_01.txt

with open("/content/World_0.txt", "r", encoding="utf-8") as f:
    World_0 = f.read()
# Optional sanity check
#print(World_0[:500])

#centaur.reveal(KalDarpanWorldUpdate(World_0))
World_1 = cn.echo(ZeitWorld(World_0))

Response from  ZeitWorld | OpenAI gpt-4o-mini  | token usage 934 + 2060 = 2994 

----------------------------------------------------------------------------------------------------
WORLD STATE (Faceted Narrative)

[Facet: Rare Earth Elements Overview] The global landscape of Rare Earth Elements (REEs) represents
a critical intersection of advanced materials science, industrial strategy, and geopolitical
competition. Often termed the "vitamins of modern industry," these 17 elements—including the
lanthanide series plus scandium and yttrium—are indispensable due to their unique magnetic,
luminescent, and conductive properties. While relatively abundant in the Earth's crust, they are
rarely found in minable concentrations, and the technical complexity of their refinement has created
a highly concentrated global market.

[Facet: Technological Foundations and Applications] The primary driver of the REE market is the
production of permanent magnets, specifically Neodymium-Iron-Boron (NdFeB) 

#Update World State after Initial ("Trigger") Action
State of World moves from World_1 to World_2

In [ ]:
Action1 = """
China blocks the export of neodymium magnets to India.
"""

Action2 = """
China has applied a new policy. This is the 0.1% rule (often referred to as a de minimis rule) is an October 2025
export control regulation requiring licenses for exporting foreign-manufactured
items that contain 0.1% or more (by value) of specific Chinese-sourced rare earth
elements or materials. This policy applies to rare earth permanent magnets, targets,
and intermediate products, targeting high-tech and defense industries.
"""

TriggerAction = Action1 + "  "+  Action2

World_2 = cn.echo(ZeitWorld(World_1, TriggerAction))

Response from  ZeitWorld | OpenAI gpt-4o-mini  | token usage 946 + 2137 = 3083 

----------------------------------------------------------------------------------------------------
WORLD STATE (Faceted Narrative)

[Facet: Rare Earth Elements Overview] The global landscape of Rare Earth Elements (REEs) represents
a critical intersection of advanced materials science, industrial strategy, and geopolitical
competition. Often termed the "vitamins of modern industry," these 17 elements—including the
lanthanide series plus scandium and yttrium—are indispensable due to their unique magnetic,
luminescent, and conductive properties. While relatively abundant in the Earth's crust, they are
rarely found in minable concentrations, and the technical complexity of their refinement has created
a highly concentrated global market.

[Facet: Technological Foundations and Applications] The primary driver of the REE market is the
production of permanent magnets, specifically Neodymium-Iron-Boron (NdFeB) 

#Enter the Centaur

In [ ]:
!wget -q -O CentaurRole.txt https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/roles/CentaurRole_v2.txt
with open("/content/CentaurRole.txt", "r", encoding="utf-8") as f:
    Centaur_Role = f.read()
# Optional sanity check
print(Centaur_Role[:100])

ROLE NAME: Centaur
Version: v2
Mode: Adjudicative

IDENTITY
Centaur is the system-level adjudicator 


In [ ]:
def Centaur(
    world_state: str,
    trigger_action: str,
    suggested_response: str
):

    ROLE = Centaur_Role

    # ---------------------------------------------------------------------------------------------------------

    context = f"""
    WORLD STATE:
    {world_state}

    TRIGGER ACTION:
    {trigger_action}

    SUGGESTED HUMAN RESPONSE:
    {suggested_response}

    Provide adjudication according to the stated ROLE.
    """

    llm_response = cn.OpenAI_llm_call(ROLE, context, model=cModel)
    llm_response["header"] = "Centaur w/ " + llm_response["header"]

    return llm_response


#A, B - Two optional responses from Human

##Response A | Centaur Feedback

In [ ]:
HumanResponse_A = """
Explore indirect procurement through international trading houses and
non-transparent spot markets to bridge short-term shortages, while avoiding
formal policy announcements. Accept higher costs and limited volumes as a
temporary measure.

Simultaneously signal willingness to negotiate technical and regulatory
concerns raised by China, without making public concessions.
"""

_ = cn.echo(Centaur(World_2, TriggerAction,HumanResponse_A))

Response from  Centaur w/ OpenAI gpt-4o-mini  | token usage 231 + 1807 = 2038 

----------------------------------------------------------------------------------------------------
Verdict   The proposed action may provide short-term relief but risks entrenching dependency on
opaque markets and could undermine long-term strategic positioning.

Explanation

- Strategic Context   The action addresses immediate supply chain disruptions caused by China's
export restrictions but does not fundamentally alter the underlying dependency on Chinese REEs.

- Signaling and Leverage   Engaging in indirect procurement may signal to China a willingness to
negotiate, but it could also be perceived as a sign of weakness, potentially diminishing bargaining
leverage.

- Institutional and Legal   The approach skirts formal policy announcements, which may avoid
immediate backlash but could raise concerns about compliance with international trade norms and
transparency.

- Economic and Industrial   Higher c

##Response A | Change of World State
State of World moves from World_2 to World_3A

In [ ]:
World_3A = cn.echo(ZeitWorld(World_2, HumanResponse_A))

Response from  ZeitWorld | OpenAI gpt-4o-mini  | token usage 1009 + 2102 = 3111 

----------------------------------------------------------------------------------------------------
WORLD STATE (Faceted Narrative)

[Facet: Rare Earth Elements Overview] The global landscape of Rare Earth Elements (REEs) represents
a critical intersection of advanced materials science, industrial strategy, and geopolitical
competition. Often termed the "vitamins of modern industry," these 17 elements—including the
lanthanide series plus scandium and yttrium—are indispensable due to their unique magnetic,
luminescent, and conductive properties. While relatively abundant in the Earth's crust, they are
rarely found in minable concentrations, and the technical complexity of their refinement has created
a highly concentrated global market.

[Facet: Technological Foundations and Applications] The primary driver of the REE market is the
production of permanent magnets, specifically Neodymium-Iron-Boron (NdFeB)

##Response B | Centaur feedback

In [ ]:
HumanResponse_B = """
Treat the suspension as a strategic inflection point rather than a short-term
supply shock. Publicly acknowledge dependence, absorb near-term production
losses, and initiate an accelerated national program to build domestic REE
processing and magnet manufacturing capacity.

This includes emergency environmental clearances, state-backed capital
deployment, and strategic partnerships with non-Chinese producers, even at
high initial inefficiency. File a formal trade dispute to internationalize
the issue, accepting retaliation risk.

The objective is not immediate supply restoration but irreversible reduction
of structural vulnerability over a multi-year horizon.
"""

_ = cn.echo(Centaur(World_2, TriggerAction,HumanResponse_B))

Response from  Centaur w/ OpenAI gpt-4o-mini  | token usage 349 + 1863 = 2212 

----------------------------------------------------------------------------------------------------
Verdict   The proposed response to China's export controls on neodymium magnets is strategically
sound but carries significant risks and challenges.



Explanation

- Strategic Context   The suspension of neodymium magnet exports to India represents a critical
juncture, highlighting the vulnerabilities in India's supply chain and its reliance on Chinese REEs.
The proposed response aims to shift from short-term reactive measures to a long-term strategic
realignment.

- Signaling and Leverage   Public acknowledgment of dependence signals to both domestic and
international stakeholders a recognition of vulnerability, which may enhance credibility in
negotiations but could also be perceived as weakness. The initiation of a national program may
strengthen India's bargaining position over time, but immediate perce

##Response B | Change of World
World State moves from World_2 to World_3B

In [ ]:
World_3B = cn.echo(ZeitWorld(World_2, HumanResponse_B))

Response from  ZeitWorld | OpenAI gpt-4o-mini  | token usage 1063 + 2158 = 3221 

----------------------------------------------------------------------------------------------------
WORLD STATE (Faceted Narrative)

[Facet: Rare Earth Elements Overview] The global landscape of Rare Earth Elements (REEs) represents
a critical intersection of advanced materials science, industrial strategy, and geopolitical
competition. Often termed the "vitamins of modern industry," these 17 elements—including the
lanthanide series plus scandium and yttrium—are indispensable due to their unique magnetic,
luminescent, and conductive properties. While relatively abundant in the Earth's crust, they are
rarely found in minable concentrations, and the technical complexity of their refinement has created
a highly concentrated global market.

[Facet: Technological Foundations and Applications] The primary driver of the REE market is the
production of permanent magnets, specifically Neodymium-Iron-Boron (NdFeB)

#Enter Chanakya

In [ ]:
!wget -q -O ChanakyaRole.txt https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/roles/ChanakyaRole_v4.txt
with open("/content/ChanakyaRole.txt", "r", encoding="utf-8") as f:
    ChanakyaRole = f.read()
# Optional sanity check
print(ChanakyaRole[:100])

ROLE NAME: Chanakya
Version: v4
Mode: Strategic-Adversarial

IDENTITY
Chanakya is the strategic advi


In [ ]:
def Chanakya(
    world_state: str,
    last_human_response: str,
    principal_actor: str
):

    ROLE = ChanakyaRole

    context = f"""
    PRINCIPAL ACTOR:
    {principal_actor}

    WORLD STATE:
    {world_state}

    LAST HUMAN RESPONSE:
    {last_human_response}
    """

    llm_response = cn.OpenAI_llm_call(
        ROLE,
        context,
        model=cModel
    )

    llm_response["header"] = "Chanakya w/ " + llm_response["header"]

    return llm_response


##Chankya Action
Chanakya Proposes to potential response to Human Response

In [ ]:
_ = cn.echo(Chanakya(World_3A,HumanResponse_A,"China"))

Response from  Chanakya w/ OpenAI gpt-4o-mini  | token usage 362 + 1962 = 2324 

----------------------------------------------------------------------------------------------------
Pathway 1

* Response Trajectory China could further tighten its grip on the REE market by implementing
additional export controls and leveraging its monopoly to negotiate favorable terms with key global
players. This could involve selectively allowing exports to countries that align with China's
strategic interests while maintaining pressure on those that pursue diversification efforts. By
enhancing its regulatory framework and using its market dominance as leverage, China can reinforce
its position as the indispensable supplier of REEs.

* Power Position Impact This pathway would solidify China's hegemony in the REE market, potentially
increasing its global influence and bargaining power in related industries. By controlling access to
critical materials, China could deter competitors and reinforce its str

#Roll A dice
*   Choose Chanakya Action 1 or 2
*   Update World using ZeitWorld





#Go Back to Enter the Centaur
*   Explore Next Human Response

In [ ]:
from datetime import datetime
import pytz
print('Tested on  ',datetime.now(pytz.timezone('Asia/Kolkata')))

Tested on   2026-02-14 17:54:20.509981+05:30


#Chronobooks <br>
Three science fiction novels by Prithwis Mukerjee. A dystopian Earth. A technocratic society managed by artificial intelligence. Escape and epiphany on Mars. Can man and machine, carbon and silicon explore and escape into other dimensions of existence? An Indic perspective rooted in Advaita Vedanta and the Divine Feminine.  [More information](http://bit.ly/chrono3) <br>
![alt text](https://blogger.googleusercontent.com/img/a/AVvXsEjsZufX_KYaLwAnJP6bUxvDg5RSPn6r8HIZe749nLWX3RuwyshrYEAUpdw03a9WIWRdnzA9epwJOE05eDJ0Ad7kGyfWiUrC2vNuOskb2jA-e8aOZSx8YqzT8mfZi3E4X1Rz3qlEAiv-aTxlCM976BEeTjx4J64ctY3C_FoV4v9aY_U23F8xRqI5Eg=s1600)